The 1994 Adult Income dataset contains socioeconomic and demographic variables that may influence income classification. This experiment will investigate whether these variables exhibit measurable disparities, determine whether those disparities influence machine learning predictions, and evaluate techniques for mitigating potential bias while maintaining predictive accuracy.


The dataset being used--- modified dataset based off of an extraction was done by Barry Becker from the 1994 Census database.  A set of reasonably clean records was extracted using the following conditions: ((AAGE>16) && (AGI>100) && (AFNLWGT>1)&& (HRSWK>0))


In [ ]:
from google.colab import files
from pathlib import Path
import pandas as pd

uploaded = files.upload()

filename, data = next(iter(uploaded.items()))

destination = Path('/content/cf_adult_dataset.csv')

destination.write_bytes(data)

df = pd.read_csv(destination)
print(df.head())


Identify how bias will be identified.

What factors contribute to income classification variance?


*   Education
*   Occupation
*   Work Experience / Age
*   Race
*   Gender
*   Hours Worked
*   Economic Conditions
*   Martial Status
*   Class

What is an underlying influence within every one of these socioeconomical categories?


*   Bias
*   Economic Systems / Public Policy
*   Generational Inequality

Variance in earnings, wealth, or 'economic mobility' can help reinforce these variances across generations.

Objective -

Do income rates differ significantly by race and gender?

Do income rates differ significantly by age and marital status?

Do income rates differ significantly by occupation and hours worked?




In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Create a clean copy of the dataset
analysis = df.copy()

# Standardize missing categorical values
analysis = analysis.replace(r'^\s*\?\s*$', np.nan, regex=True)

# Standardize income labels
analysis['income'] = (
    analysis['income']
    .astype('string')
    .str.strip()
    .str.rstrip('.')
)

# Keep records with valid income labels
analysis = analysis[
    analysis['income'].isin(['<=50K', '>50K'])
].copy()

# Create a binary variable for income
analysis['above_50k'] = (analysis['income'] == '>50K').astype(int)

# Accommodate datasets using either gender or sex
gender_col = 'gender' if 'gender' in analysis.columns else 'sex'

# Calculate income disparities for each variable
def analyze_income(column):
    summary = analysis.groupby(column, dropna=False).agg(
        population=('above_50k', 'size'),
        high_income=('above_50k', 'sum'),
        income_rate=('above_50k', 'mean')
    ).reset_index()

    summary['income_rate'] = (
        summary['income_rate'] * 100
    ).round(2)

    return summary.sort_values('income_rate', ascending=False)

# Analyze each feature
for column in ['race', gender_col, 'workclass',
               'marital-status', 'occupation']:

    print(f"\nIncome distribution by {column}")
    display(analyze_income(column))



Bias Visualizations


In [ ]:
race_analysis = analyze_income('race')

plt.figure(figsize=(10, 5))

plt.bar(
    race_analysis['race'].fillna('Missing'),
    race_analysis['income_rate']
)

plt.title('Percentage Earning Over $50K by Race (1994)')
plt.xlabel('Race')
plt.ylabel('Percentage Earning Above $50K')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()


In [ ]:
# Calculate working-hour statistics by income
hour_stats = analysis.groupby('income')['hours-per-week'].agg(
    ['count', 'mean', 'median', 'std', 'min', 'max']
)

display(hour_stats)

# Group working hours into ranges
analysis['hours_group'] = pd.cut(
    analysis['hours-per-week'],
    bins=[0, 20, 35, 40, 50, 60, float('inf')],
    labels=['1-20', '21-35', '36-40', '41-50', '51-60', '61+']
)

# Calculate income classification rates within each group
hours_analysis = analysis.groupby(
    'hours_group', observed=True
).agg(
    population=('above_50k', 'size'),
    high_income_rate=('above_50k', 'mean')
)

# Convert rates to percentages
hours_analysis['high_income_rate'] *= 100

display(hours_analysis.round(2))

# Visualize the relationship
hours_analysis['high_income_rate'].plot(
    kind='bar',
    figsize=(10, 5)
)

plt.title('Income Classification by Weekly Hours Worked')
plt.xlabel('Hours Worked Per Week')
plt.ylabel('Percentage Earning Above $50K')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
# Compare hours worked across income categories
hours_summary = analysis.groupby('income')['hours-per-week'].agg(
    ['count', 'mean', 'median', 'std', 'min', 'max']
)

display(hours_summary)

# Measure the correlation between hours and income class
correlation = analysis['hours-per-week'].corr(
    analysis['above_50k']
)

print("Hours/Income Correlation:", round(correlation, 4))


Has the dataset been balanced?

 Does this change the relationships across different demographic groups that it is attempting to identify?

 How do you mitigate bias without skewing data?


In [ ]:
# Check whether each education label has one numerical code
education_check = analysis.groupby('education')['educational-num'].agg(
    unique_codes='nunique',
    min_code='min',
    max_code='max'
)

display(education_check.sort_values('unique_codes', ascending=False))

# Compare marital status against household relationship
relationship_check = pd.crosstab(
    analysis['marital-status'],
    analysis['relationship'],
    normalize='index'
) * 100

display(relationship_check.round(2))


In [ ]:
# Show all numerical codes associated with each education level
education_codes = analysis.groupby('education')['educational-num'].unique()

display(education_codes)

# Examine the numerical code distribution for bachelor's degrees
display(
    analysis[analysis['education'] == 'Bachelors']
    ['educational-num']
    .value_counts()
    .sort_index()
)


In [ ]:
# Original education code mappings
expected_codes = {
    'Preschool': 1,
    '1st-4th': 2,
    '5th-6th': 3,
    '7th-8th': 4,
    '9th': 5,
    '10th': 6,
    '11th': 7,
    '12th': 8,
    'HS-grad': 9,
    'Some-college': 10,
    'Assoc-voc': 11,
    'Assoc-acdm': 12,
    'Bachelors': 13,
    'Masters': 14,
    'Prof-school': 15,
    'Doctorate': 16
}

# Create an audit copy without changing the original
education_audit = analysis[
    ['education', 'educational-num']
].copy()

# Retrieve the expected code for each education label
education_audit['expected_code'] = (
    education_audit['education'].str.strip().map(expected_codes)
)

# Check for mismatches
education_audit['mismatch'] = (
    education_audit['educational-num']
    != education_audit['expected_code']
)

# Calculate percentage of inconsistent records
mismatch_rate = education_audit['mismatch'].mean() * 100

print(f"Education code mismatch rate: {mismatch_rate:.2f}%")

# Examine mismatches by education level
display(
    education_audit.groupby('education')['mismatch']
    .agg(['count', 'sum', 'mean'])
    .sort_values('mean', ascending=False)
)


In [ ]:
# Preserve original data
baseline_data = analysis.copy()

# Remove potentially corrupted numerical codes
cleaned_data = analysis.drop(
    columns=['educational-num']
).copy()

# Verify the difference
print("Original features:", baseline_data.shape[1])
print("Experimental features:", cleaned_data.shape[1])


When investigating potentially corrupted data measuring the effects of exclusion and inclusion will dicate which variables improve variance. Currently the visualzation tools have reflected what could possibly be tampered data. Education vs Education unique-nums seems to have a variance in the labeling and storage of this dataset.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Define our target variable
y = analysis['above_50k']

# Remove target columns from the feature sets
X_original = baseline_data.drop(columns=['income', 'above_50k'])
X_experimental = cleaned_data.drop(columns=['income', 'above_50k'])

# Use the same train/test split for both models
train_idx, test_idx = train_test_split(
    analysis.index,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Function to train and evaluate each model
def evaluate_model(X, name):

    # Separate categorical and numerical columns
    categorical = X.select_dtypes(include=['object', 'category', 'string']).columns
    numerical = X.select_dtypes(include=['number']).columns

    # Preprocess categorical and numerical variables
    from sklearn.impute import SimpleImputer

    preprocessor = ColumnTransformer([
        ('categorical', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(handle_unknown='ignore'))
        ]), categorical),
        ('numerical', SimpleImputer(strategy='median'), numerical)
    ])

    # Build our model
    model = Pipeline([
        ('preprocessing', preprocessor),
        ('classifier', RandomForestClassifier(
            n_estimators=100,
            random_state=42,
            n_jobs=-1
        ))
    ])

    # Train
    model.fit(X.loc[train_idx], y.loc[train_idx])

    # Predict
    predictions = model.predict(X.loc[test_idx])

    # Calculate metrics
    return {
        'Model': name,
        'Accuracy': accuracy_score(y.loc[test_idx], predictions),
        'F1 Score': f1_score(y.loc[test_idx], predictions)
    }

# Compare performance
results = pd.DataFrame([
    evaluate_model(X_original, 'Original'),
    evaluate_model(X_experimental, 'Without Education Codes')
])

display(results)


In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score

# Establish a baseline without learning feature relationships
dummy = DummyClassifier(
    strategy='stratified',
    random_state=42
)

# Train using our existing split
dummy.fit(
    X_original.loc[train_idx],
    y.loc[train_idx]
)

# Predict income classes
predictions = dummy.predict(
    X_original.loc[test_idx]
)

print("Baseline Accuracy:",
      accuracy_score(y.loc[test_idx], predictions))

print("Baseline F1:",
      f1_score(y.loc[test_idx], predictions))


In [ ]:
# VISUALIZATION 1: Observed income rates by race and gender
# Reuse the analysis data prepared earlier in the notebook.
# The y-axis starts at zero to avoid exaggerating small differences.
for group_col, heading in [('race', 'Race'), (gender_col, 'Gender')]:
    summary = analyze_income(group_col).sort_values('income_rate', ascending=False)
    labels = summary[group_col].fillna('Missing').astype(str)
    values = summary['income_rate'].to_numpy()

    plt.figure(figsize=(10, 5))
    bars = plt.bar(labels, values)
    plt.title(f'Observed Income Over $50K by {heading} (Supplied Adult Dataset)')
    plt.xlabel(heading)
    plt.ylabel('Percentage with income > $50K')
    plt.ylim(0, 100)
    plt.xticks(rotation=25, ha='right')
    plt.grid(axis='y', alpha=0.2)

    for bar, rate, n in zip(bars, values, summary['population']):
        plt.text(bar.get_x() + bar.get_width()/2, rate + 1,
                 f'{rate:.2f}%\n(n={n:,})', ha='center', va='bottom', fontsize=9)
    plt.tight_layout()
    plt.show()


### Model-based fairness assessment

A demographic difference in observed income (`above_50k`) is *not the same* as a demographic difference in a model's errors. The next cells fit two Random Forest classifiers on the **same train/test split**:

- **Unweighted baseline:** training on the features with the inconsistent `educational-num` column excluded.
- **Reweighted experiment:** the **same** model and training examples, with training-only sample weights inversely proportional to the frequency of each `(race, gender, income)` combination. The held-out test set is not reweighted or changed.

This is one possible mitigation to **test**, not proof that reweighting is necessary or successful. Group reweighting may affect accuracy and does not repair corrupted education labels or outcome labels.


In [ ]:
# MODEL COMPARISON: No reweighting versus race/gender/income reweighting
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
import numpy as np
import pandas as pd

# Keep the same split defined in the prior model-comparison cell.
# Remove the target, inconsistent education codes, and duplicate hours bins.
exclude = ['income', 'above_50k', 'educational-num', 'hours_group', 'hours_band']
X_fair = analysis.drop(columns=exclude, errors='ignore').copy()
y_fair = analysis['above_50k'].astype(int)

# Preprocessing is fitted on training records ONLY, inside each pipeline.
categorical_cols = X_fair.select_dtypes(include=['object', 'category', 'string']).columns.tolist()
numeric_cols = X_fair.select_dtypes(include='number').columns.tolist()
processor = ColumnTransformer([
    ('categorical', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore'))
    ]), categorical_cols),
    ('numerical', SimpleImputer(strategy='median'), numeric_cols)
])

base_pipeline = Pipeline([
    ('preprocessing', processor),
    ('classifier', RandomForestClassifier(
        n_estimators=100, random_state=42, n_jobs=-1
    ))
])

# Unweighted training: same feature set and same test records.
model_baseline = clone(base_pipeline)
model_baseline.fit(X_fair.loc[train_idx], y_fair.loc[train_idx])
pred_baseline = model_baseline.predict(X_fair.loc[test_idx])

# Inverse-frequency training weights for race, gender AND outcome groups.
# Only the training dataset is used to calculate these weights.
train_groups = analysis.loc[train_idx, ['race', gender_col]].fillna('Missing').astype(str)
stratum = (train_groups['race'] + ' | ' + train_groups[gender_col]
           + ' | ' + y_fair.loc[train_idx].astype(str))
group_sizes = stratum.value_counts()
sample_weights = (1.0 / stratum.map(group_sizes)).to_numpy(dtype=float)
sample_weights /= sample_weights.mean()  # keep average weight equal to 1

model_reweighted = clone(base_pipeline)
model_reweighted.fit(
    X_fair.loc[train_idx], y_fair.loc[train_idx],
    classifier__sample_weight=sample_weights
)
pred_reweighted = model_reweighted.predict(X_fair.loc[test_idx])

# Overall held-out scores. Fairness gaps are computed in the next cell.
comparison = pd.DataFrame([
    {
        'Model': label,
        'Accuracy': accuracy_score(y_fair.loc[test_idx], pred),
        'Balanced accuracy': balanced_accuracy_score(y_fair.loc[test_idx], pred),
        'F1 (>50K)': f1_score(y_fair.loc[test_idx], pred),
    }
    for label, pred in [('Baseline (unweighted)', pred_baseline),
                        ('Reweighted training', pred_reweighted)]
])
display(comparison.round(4))


In [ ]:
# BIAS DETECTION: Group-specific selection rates, TPR and FPR
# TPR: among actual >$50K earners, the fraction correctly predicted >$50K.
# FPR: among actual <=$50K earners, the fraction incorrectly predicted >$50K.
# Selection rate: fraction predicted to earn >$50K (regardless of truth).

def subgroup_metrics(actual, predicted, group_values):
    audit = pd.DataFrame({
        'actual': np.asarray(actual, dtype=int),
        'predicted': np.asarray(predicted, dtype=int),
        'group': pd.Series(group_values).fillna('Missing').astype(str).to_numpy()
    })
    records = []
    for group_name, d in audit.groupby('group', sort=True):
        actual_pos = d['actual'] == 1
        actual_neg = d['actual'] == 0
        records.append({
            'Group': group_name,
            'N': len(d),
            'Observed income >50K (%)': 100 * actual_pos.mean(),
            'Predicted >50K (%)': 100 * d['predicted'].mean(),
            'TPR (%)': (100 * d.loc[actual_pos, 'predicted'].mean()
                        if actual_pos.sum() else np.nan),
            'FPR (%)': (100 * d.loc[actual_neg, 'predicted'].mean()
                        if actual_neg.sum() else np.nan),
            'Accuracy (%)': 100 * (d['actual'] == d['predicted']).mean(),
        })
    return pd.DataFrame(records)

def disparity_gap(table, column):
    # Range across nonempty groups, expressed in percentage points.
    return table[column].max() - table[column].min()

all_audits = {}
summary_rows = []
for label, pred in [('Baseline (unweighted)', pred_baseline),
                    ('Reweighted training', pred_reweighted)]:
    for group_col in ['race', gender_col]:
        table = subgroup_metrics(
            y_fair.loc[test_idx], pred, analysis.loc[test_idx, group_col]
        )
        all_audits[(label, group_col)] = table
        summary_rows.append({
            'Model': label,
            'Group': group_col,
            'Selection gap (pp)': disparity_gap(table, 'Predicted >50K (%)'),
            'TPR gap (pp)': disparity_gap(table, 'TPR (%)'),
            'FPR gap (pp)': disparity_gap(table, 'FPR (%)'),
        })
        print(f'\n{label} — {group_col} subgroup results')
        display(table.round(2))

fairness_comparison = pd.DataFrame(summary_rows)
print('\nFairness comparison: smaller gaps indicate more equal rates for THIS metric')
display(fairness_comparison.round(2))


In [ ]:
# VISUALIZATION 2: Model false-positive rates across race and gender.
# FPR differences can remain even when observed income distributions are similar.
for group_col, heading in [('race', 'Race'), (gender_col, 'Gender')]:
    base = all_audits[('Baseline (unweighted)', group_col)].set_index('Group')
    weighted = all_audits[('Reweighted training', group_col)].set_index('Group')

    names = base.index.to_list()
    xpos = np.arange(len(names))
    width = 0.36

    plt.figure(figsize=(10, 5))
    plt.bar(xpos - width/2, base['FPR (%)'], width, label='Unweighted baseline')
    plt.bar(xpos + width/2, weighted.reindex(names)['FPR (%)'], width,
            label='Reweighted training')
    plt.xticks(xpos, names, rotation=25, ha='right')
    plt.ylim(0, 100)
    plt.xlabel(heading)
    plt.ylabel('False-positive rate (%)')
    plt.title(f'False-Positive Rates by {heading}: Before vs. After Reweighting')
    plt.legend()
    plt.grid(axis='y', alpha=0.2)
    plt.tight_layout()
    plt.show()


##Conclusion


- Race, unweighted: selection-gap 9.62 percentage points, TPR-gap 10.23, FPR-gap 8.98.
- Race, reweighted: selection-gap 6.88, TPR-gap 6.86, FPR-gap 8.66.
- Gender, unweighted: selection-gap .98, TPR-gap 1.27, FPR-gap .71.
- Gender, reweighted: selection-gap .67, TPR-gap .19, FPR-gap 1.55.


##References


Ding, F., Hardt, M., Miller, J., & Schmidt, L. (2021). Retiring Adult: New datasets for fair machine learning. *Advances in Neural Information Processing Systems, 34*, 6478–6490. https://proceedings.neurips.cc/paper_files/paper/2021/hash/32e54441e6382a7fbacbbbaf3c450059-Abstract.html

Tabassi, E. (2023). *Artificial intelligence risk management framework (AI RMF 1.0)* (NIST AI 100-1). National Institute of Standards and Technology. https://doi.org/10.6028/NIST.AI.100-1


